# Multi-Agent Coordination using Tabular Q-Learning

## Project Overview

This project implements a **Multi-Agent Reinforcement Learning (MARL)** simulation using **Tabular Q-Learning** in a 5×5 Grid World. The objective is to train two autonomous agents to coordinate their actions while operating in a shared, partially observable environment.

The environment contains two home locations (X and Y), two sample collection sites (U and V), and a lake located at the intersection of the agents' shortest paths. The lake can randomly switch between **dry** and **flooded** states during the simulation.

Two different types of agents operate within this environment:

* **Type A:** Travels between X and U. This agent is **not waterproof** and should learn to cross the lake only when it is dry.
* **Type B:** Travels between Y and V. This agent is **waterproof** and should learn a policy that avoids collisions with Type A while successfully completing its own mission.

Both agents learn independently using **Tabular Q-Learning** while interacting simultaneously with the same environment. Each agent only observes its own position, whether it is carrying a sample, and the current state of the lake. The agents cannot observe each other's locations or communicate directly.

The goal of this project is to demonstrate that coordination can emerge through reinforcement learning, allowing the agents to complete their tasks efficiently while minimizing collisions and unnecessary penalties.

---

## Objectives

The objectives of this project are:

* Build a multi-agent Grid World environment.
* Implement simultaneous action execution for multiple agents.
* Train both agents using independent Tabular Q-Learning.
* Model dynamic lake behavior with probabilistic state changes.
* Implement collision detection and environment penalties.
* Analyze the learning performance using training statistics and visualizations.
* Demonstrate the emergence of coordinated behavior after training.

---

## Environment Summary

* Grid Size: **5 × 5**
* Agents: **2 (Type A and Type B)**
* Action Space: **North, South, East, West, Wait**
* Learning Algorithm: **Tabular Q-Learning**
* Observation: Agent position, carrying status, and lake state
* Environment: Partially Observable
* Lake State: Dry or Flooded (changes probabilistically)

---

## Expected Learning Behavior

After sufficient training, the learned policy should exhibit the following behavior:

* Type A crosses the lake only when it is dry.
* Type B coordinates its behavior to minimize collisions with Type A.
* Both agents successfully collect and deliver samples.
* Collision frequency decreases significantly over time.
* Overall cumulative reward and task completion rate improve as training progresses.

---

## Notebook Structure

This notebook is organized into the following sections:

1. Python libraries
2. Project Configurations
3. Environment Implementation
4. Agent Representation
5. Q-Learning Implementation
6. Training Process
7. Performance Evaluation
8. Visualization
9. Demonstration of the Learned Policy


## Python Libraries
This section defines all necessary python libraries, including

* numpy
* matplotlib.pyplot
* dataclasses

In [37]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

## Project Configurations

This section defines all configuration parameters required throughout the project, including:

* Environment configuration
* Training configuration
* Reward configuration
* Q-Learning configuration
* Lake configuration
* Action space


In [38]:
# ==========================================
# Environment Configuration
# ==========================================

@dataclass
class EnvironmentConfig:
    # Grid size
    grid_size: int = 5

    # Agent A locations
    home_a: tuple = (4, 0)
    pickup_a: tuple = (2, 4)

    # Agent B locations
    home_b: tuple = (0, 2)
    pickup_b: tuple = (4, 2)

    # Lake location
    lake_position: tuple = (2, 2)


# ==========================================
# Training Configuration
# ==========================================

@dataclass
class TrainingConfig:
    # Number of training episodes
    episodes: int = 10000

    # Maximum steps per episode
    max_steps: int = 100


# ==========================================
# Reward Configuration
# ==========================================

@dataclass
class RewardConfig:
    # Movement rewards
    step_reward: int = -5
    wait_reward: int = -3

    # Penalties
    collision_penalty: int = -20
    water_penalty: int = -20

    # Mission rewards
    pickup_reward: int = 10
    delivery_reward: int = 50


# ==========================================
# Q-Learning Configuration
# ==========================================

@dataclass
class QLearningConfig:
    # Learning parameters
    learning_rate: float = 0.1
    discount_factor: float = 0.95

    # Exploration parameters
    epsilon: float = 1.0
    epsilon_decay: float = 0.995
    min_epsilon: float = 0.01


# ==========================================
# Lake Configuration
# ==========================================

@dataclass
class LakeConfig:
    # Probability that the lake changes state
    flip_probability: float = 0.20


# ==========================================
# Training statestics
# ==========================================

@dataclass
class TrainingStates:
    ep_rewards: list
    ep_steps: list
    ep_done: list
    epsilon_history: list

    def __init__(self):
        self.ep_rewards = []
        self.ep_steps = []
        self.ep_done = []
        self.epsilon_history = []

# ==========================================
# Create Configuration Objects
# ==========================================

environment = EnvironmentConfig()
training = TrainingConfig()
rewards = RewardConfig()
q_learning = QLearningConfig()
lake = LakeConfig()
training_states = TrainingStates()

# ==========================================
# Action Space
# ==========================================

# (row change, column change)
ACTIONS = [
    (-1, 0),   # North
    (1, 0),    # South
    (0, -1),   # West
    (0, 1),    # East
    (0, 0)     # Wait
]

# Action names for visualization
ACTION_NAMES = [
    "North",
    "South",
    "West",
    "East",
    "Wait"
]

## Environment Implementation

This section implements the Grid World environment, including:

* Agent representation
* Environment initialization
* Environment reset
* Agent observations
* Agent movement
* Simultaneous action execution
* Pickup and delivery mechanics
* Collision detection
* Water damage handling
* Lake state updates
* Environment rendering


### Agent representation

In [39]:
@dataclass
class Agent:
    home: tuple                  # Home location
    pickup: tuple                # Sample pickup location
    is_waterproof: bool          # Whether the agent is waterproof
    curr_position: tuple = None  # Current position
    prev_position: tuple = None  # previous position
    is_carrying: bool = False    # Whether the agent is carrying a sample
    is_delivered: bool = False   # Whether the delivery is completed

### GridWorld

This section implements the **GridWorld** environment, which manages the interaction between both agents and the environment. It provides the following functionalities:

* Environment initialization
* Episode reset
* Agent state observation
* Simultaneous action execution
* Agent movement
* Pickup and delivery handling
* Collision detection
* Water damage handling
* Lake state updates
* Environment rendering


In [40]:
class GridWorld:
# ===============================================================================
# Environment initialize
# ===============================================================================
    
    def __init__(self):
        self.size = environment.grid_size
        self.lake_position = environment.lake_position

        # Create Agent A
        self.agent_a = Agent(
            home=environment.home_a,
            pickup=environment.pickup_a,
            is_waterproof=False
        )

        # Create Agent B
        self.agent_b = Agent(
            home=environment.home_b,
            pickup=environment.pickup_b,
            is_waterproof=True
        )

        # Start the first episode
        self.reset()

    # Start a new episode
    def reset(self):
        self.reset_agent(self.agent_a)
        self.reset_agent(self.agent_b)
        self.initialize_lake()
        
        return self.get_state_of_both()

# ===================================================================================
# Episode Reset
# ===================================================================================
    
    def reset_agent(self, agent):
        agent.curr_position = agent.home
        agent.is_carrying = False
        agent.is_delivered = False

    # Randomly initialize the lake state
    def initialize_lake(self):
        self.is_lake_flooded = np.random.choice([True, False])

# ===================================================================================
# Agent State Observation
# ===================================================================================

    # Return one agent's observation
    def get_state_of(self, agent):
        return (
            agent.curr_position,
            agent.is_carrying,
            self.is_lake_flooded
        )

    # Return observations for both agents
    def get_state_of_both(self):
        return (
            self.get_state_of(self.agent_a),
            self.get_state_of(self.agent_b)
        )

# ====================================================================================
# Agent movement
# ====================================================================================
    
    # Check whether both agents completed delivery
    def is_mission_completed(self):
        a_done = self.agent_a.is_delivered
        b_done = self.agent_b.is_delivered

        return a_done and b_done

######################################################################################3
    
    # Update the lake state
    def update_lake(self):
        chance = np.random.random()
    
        if chance > lake.flip_probability:
            return
    
        self.is_lake_flooded = (not self.is_lake_flooded)
        
#################################################################################################
    
    # Check whether both agents collided
    def is_collided(self):
        same_position = (
            self.agent_a.curr_position ==
            self.agent_b.curr_position
        )
        
        in_lake = (
            self.agent_a.curr_position ==
            self.lake_position
        )
        
        return same_position and in_lake


    # Return collision penalty
    def collision_penalty(self):
        collided = self.is_collided()
    
        if collided:
            return rewards.collision_penalty
    
        return 0

##################################################################################################

    # Check whether an agent is in the lake
    def is_in_lake(self, agent):
        return (agent.curr_position == self.lake_position)

    
    # Return water damage penalty
    def water_penalty(self, agent):
        if agent.is_waterproof:
            return 0
    
        if self.is_lake_flooded == False:
            return 0
            
        in_lake = self.is_in_lake(agent)
        if in_lake:
            return rewards.water_penalty
    
        return 0
        
#################################################################################################3
    
    #return delivery reward
    def deliver_sample(self, agent):
        at_home = agent.curr_position == agent.home
        delivered = agent.is_carrying and at_home
        
        if delivered:
            agent.is_carrying = False
            agent.is_delivered = True
            return rewards.delivery_reward
        return 0
    
################################################################################3
        
    # Return pickup reward
    def pickup_sample(self, agent):
        if agent.is_carrying:
            return 0
            
        at_pickup = agent.curr_position == agent.pickup

        if at_pickup:
            agent.is_carrying = True
            return rewards.pickup_reward
        return 0

###################################################################################3
    
    # Return movement reward
    def action_reward(self, action):
        #if action is wait
        if action == 4:
            return rewards.wait_reward

        return rewards.step_reward

######################################################################################
    
    # Calculate one agent's reward
    def calculate_reward(self, agent, action):
        reward = self.action_reward(action)
        reward += self.pickup_sample(agent)
        reward += self.deliver_sample(agent)
        reward += self.water_penalty(agent)
        reward += self.collision_penalty()

        return reward

################################################################################### move_agent
        
    # Check whether a position is inside the grid
    def is_inside_grid(self, row, col):
        valid_row = 0 <= row < self.size
        valid_col = 0 <= col < self.size
        return valid_row and valid_col


    # Return the next valid position
    def next_position(self, position, action):
        dx, dy = ACTIONS[action]

        row = position[0] + dx
        col = position[1] + dy

        if self.is_inside_grid(row, col):
            return (row, col)

        return position


    # Move one agent
    def move_agent(self, agent, action):
        next_pos = self.next_position(agent.curr_position, action)
        agent.curr_position = next_pos

################################################################################### execute_action
        
    # Execute actions for both agents
    def execute_action(self, action_a, action_b):
        self.move_agent(self.agent_a, action_a)
        self.move_agent(self.agent_b, action_b)
        
        #reward calculation
        reward_a = self.calculate_reward(self.agent_a, action_a)
        reward_b = self.calculate_reward(self.agent_b, action_b)

        self.update_lake()

        return (
            self.get_state_of(self.agent_a),
            self.get_state_of(self.agent_b),
            reward_a,
            reward_b,
            self.is_mission_completed()
        )

# ====================================================================================
# Agent movement
# ====================================================================================

    # Create an empty grid
    def create_grid(self):
        return [
            ["." for _ in range(self.size)]
            for _ in range(self.size)
        ]
    
    
    # Place fixed locations
    def place_locations(self, grid):
        grid[self.agent_a.home[0]][self.agent_a.home[1]] = "X"
        grid[self.agent_b.home[0]][self.agent_b.home[1]] = "Y"
    
        grid[self.agent_a.pickup[0]][self.agent_a.pickup[1]] = "U"
        grid[self.agent_b.pickup[0]][self.agent_b.pickup[1]] = "V"
    
        row, col = self.lake_position
        grid[row][col] = "L"
    
    
    # Place both agents
    def place_agents(self, grid):
        a_row, a_col = self.agent_a.curr_position
        b_row, b_col = self.agent_b.curr_position
    
        if self.agent_a.curr_position == self.agent_b.curr_position:
            grid[a_row][a_col] = "*"
            return
    
        grid[a_row][a_col] = "A"
        grid[b_row][b_col] = "B"
    
    
    # Print the environment
    def render(self):
        grid = self.create_grid()
    
        self.place_locations(grid)
        self.place_agents(grid)
    
        print()
    
        for row in grid:
            print(" ".join(row))
    
        print()
        print(f"Lake      : {'Flooded' if self.is_lake_flooded else 'Dry'}")
        print(f"A Carrying: {self.agent_a.is_carrying}")
        print(f"B Carrying: {self.agent_b.is_carrying}")
        print(f"A Done    : {self.agent_a.is_delivered}")
        print(f"B Done    : {self.agent_b.is_delivered}")

# Check the world

In [41]:
# # Create environment
# env = GridWorld()

# # Reset environment
# state_a, state_b = env.reset()

# env.render()

# print("Initial State")
# print("Agent A:", state_a)
# print("Agent B:", state_b)
# print("Lake Flooded:", env.is_lake_flooded)
# print()

# # Run a few random steps
# for step in range(10):

#     action_a = random.randint(0, 4)
#     action_b = random.randint(0, 4)

#     (
#         state_a,
#         state_b,
#         reward_a,
#         reward_b,
#         done
#     ) = env.execute_action(
#         action_a,
#         action_b
#     )

#     print(f"Step {step + 1}")
#     print(f"A Action : {action_a}")
#     print(f"B Action : {action_b}")
#     print(f"A State  : {state_a}")
#     print(f"B State  : {state_b}")
#     print(f"A Reward : {reward_a}")
#     print(f"B Reward : {reward_b}")
#     print(f"Lake     : {env.is_lake_flooded}")
#     print(f"Done     : {done}")
#     print("-" * 40)

#     if done:
#         print("Mission Completed!")
#         break

### Phase 3 – Q-Learning Implementation

This section implements the Q-learning algorithm for both agents. Each agent maintains its own Q-table and learns independently from its observations and rewards. Through repeated interaction with the environment, the agents learn policies that maximize their cumulative rewards while coordinating their behavior.

The implementation includes:

* Q-table initialization
* ε-greedy action selection
* Bellman equation (Q-value update)


#### Q-table initialization

In [42]:
# Q-table for Agent A
q_table_a = {}

# Q-table for Agent B
q_table_b = {}

# Initialize one state
def initialize_state(q_table, state):
    if state in q_table:
        return

    q_table[state] = np.zeros(len(ACTIONS))

#### Epsilon-greedy action selection

In [43]:
# Choose an action using ε-greedy policy
def epsilon_greedy(q_table, state, epsilon):

    initialize_state(q_table, state)

    choice = random.random()

    if choice <= epsilon:
        return np.random.randint(len(ACTIONS))

    return np.argmax(q_table[state])

####  Update Q-Table using bellman equation

In [44]:
# Update one Q-value
def update_q_table(q_table, state, action, reward, next_state):
    initialize_state(q_table, next_state)

    current_q = q_table[state][action]

    best_next_q = np.max(q_table[next_state])
    
    TD_error = (reward + q_learning.gamma * best_next_q) - current_q

    q_table[state][action] = (current_q + q_learning.alpha * TD_error)